# Lab 3: Building & Managing Datasets

## Difficulty: Beginner | ~40 min | Requires LangChain Lab 3 (Structured Output)

Learn how LangSmith datasets are structured (input/output example pairs), and the different ways to create them: manually via the SDK, by converting traces, or by importing a CSV.

In [1]:
# Install all required packages at pinned versions for reproducibility
!pip install -qU "langsmith>=0.1.0" "langchain==1.2.15" "langchain-core==1.2.28" "langchain-openai==1.1.12" "python-dotenv==1.2.2" "pydantic==2.13.4" "pandas>=2.0.0"


[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


This installs the exact versions of every library used in this lab.

In [2]:
import os
from dotenv import load_dotenv
from langsmith import Client

# Load API keys from .env file
load_dotenv()

# Verify required keys are present — fail early with a clear message
assert os.getenv("OPENROUTER_API_KEY"), "Missing OPENROUTER_API_KEY in .env"
assert os.getenv("LANGSMITH_API_KEY"), "Missing LANGSMITH_API_KEY in .env"

# Initialize the LangSmith client (talks to LangSmith's servers)
ls_client = Client()
print("Environment loaded and LangSmith client initialized")

Environment loaded and LangSmith client initialized


Loads API keys from `.env` and initializes the LangSmith client — this is what talks to LangSmith's servers for dataset operations.

In [3]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# Create the LLM client pointing to OpenRouter (free tier)
model = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,  # Deterministic output for consistent dataset entries
)

# Schema that defines what structured data we want from each review
class ProductReview(BaseModel):
    product: str = Field(description="The exact product being reviewed")
    rating: int = Field(description="The star rating, from 1 to 5")
    sentiment: str = Field(description="positive, negative, or neutral")

This sets up the structured-output agent from LangChain Lab 3. The `ProductReview` schema defines the shape we want for each example in our dataset.

In [4]:
# 10 varied product reviews covering different ratings and sentiments
review_texts = [
    "I bought the 'AeroPress Coffee Maker' two weeks ago. Best coffee ever. Five stars.",
    "The 'Ergonomic Office Chair' arrived broken. Customer service never replied. One star.",
    "My 'Wireless Noise-Cancelling Headphones' are great value. Four stars.",
    "The 'Smart Fitness Tracker' stopped working after three days. Two stars.",
    "Love the 'Portable Bluetooth Speaker' — loud, clear, waterproof. Five stars.",
    "The 'Organic Green Tea' tastes stale. Box was damaged. Two stars.",
    "My 'Mechanical Keyboard' is a dream to type on. Five stars.",
    "The 'Bamboo Cutting Board' cracked after one week. One star.",
    "Great 'LED Desk Lamp' — adjustable brightness, USB port. Four stars.",
    "My 'Running Shoes' are lightweight and supportive. Five stars.",
]

# Bind the Pydantic schema to the model so it returns ProductReview objects
structured_model = model.with_structured_output(ProductReview)

# Run the structured-output agent on each review to extract fields
parsed_reviews = []
for review_text in review_texts:
    parsed = structured_model.invoke(review_text)
    parsed_reviews.append(parsed)

# Preview the first 5 parsed results
for p in parsed_reviews[:5]:
    print(f"{p.product} | rating {p.rating}/5 | {p.sentiment}")
print(f"\nParsed {len(parsed_reviews)} reviews into structured objects")

AeroPress Coffee Maker | rating 5/5 | positive
Ergonomic Office Chair | rating 1/5 | Frustrated, unresolved issue with broken item and no customer service response
Wireless Noise-Cancelling Headphones | rating 4/5 | positive
Smart Fitness Tracker | rating 2/5 | negative_short_lifespan
Portable Bluetooth Speaker | rating 5/5 | positive

Parsed 10 reviews into structured objects


These 10 reviews cover a range of products, ratings (1-5), and sentiments. Running the structured-output agent on each produces typed `ProductReview` objects — the ground-truth data for our dataset.

In [5]:
# Clean up any existing datasets with these names (idempotent re-runs)
for ds_name in ["product-reviews", "product-reviews-from-traces", "product-reviews-csv"]:
    try:
        existing = ls_client.read_dataset(dataset_name=ds_name)
        ls_client.delete_dataset(dataset_id=existing.id)
        print(f"  Deleted existing dataset: {ds_name}")
    except Exception:
        pass  # Dataset doesn't exist yet, nothing to delete

# Create a new empty dataset in LangSmith
dataset = ls_client.create_dataset(
    dataset_name="product-reviews",
    description="Product reviews with structured output (product, rating, sentiment)"
)

# Batch-add all examples at once: inputs and outputs are lists of dicts
ls_client.create_examples(
    dataset_id=dataset.id,
    inputs=[{"review_text": rt} for rt in review_texts],      # raw review as input
    outputs=[p.model_dump() for p in parsed_reviews],          # structured output as expected answer
)

print(f"Created dataset '{dataset.name}' with {len(parsed_reviews)} examples")

  Deleted existing dataset: product-reviews


  Deleted existing dataset: product-reviews-from-traces


  Deleted existing dataset: product-reviews-csv


Created dataset 'product-reviews' with 10 examples


This creates an empty dataset in LangSmith, then adds each review as an example: the raw review text as input, the structured output as the expected answer. This input/output pairing is the foundation for evaluation.

In [6]:
# Manually defined inputs and expected outputs (no LLM needed)
manual_inputs = [
    {"review_text": "The 'Premium Blender' is fantastic. Five stars."},
    {"review_text": "My 'Laptop Stand' wobbles constantly. Two stars."},
    {"review_text": "The 'Ceramic Mug Set' looks nice but one arrived chipped. Three stars."},
]
manual_outputs = [
    {"product": "Premium Blender", "rating": 5, "sentiment": "positive"},
    {"product": "Laptop Stand", "rating": 2, "sentiment": "negative"},
    {"product": "Ceramic Mug Set", "rating": 3, "sentiment": "neutral"},
]

# Add them to the same dataset using create_examples()
ls_client.create_examples(
    dataset_id=dataset.id,
    inputs=manual_inputs,
    outputs=manual_outputs,
)
print(f"Added {len(manual_inputs)} manual examples. Total: {len(parsed_reviews) + len(manual_inputs)}")

Added 3 manual examples. Total: 13


This is the manual approach: you write the inputs and expected outputs yourself. Useful when you have existing ground-truth data or want to add specific edge cases.

In [7]:
# Query existing LLM traces from LangSmith (these were captured during Step 4)
traces = list(ls_client.list_runs(
    project_name=os.getenv("LANGSMITH_PROJECT"), run_type="llm", limit=20
))

# Create a separate dataset for the trace-based examples
trace_dataset = ls_client.create_dataset(
    dataset_name="product-reviews-from-traces",
    description="Dataset created by converting LangSmith traces into examples"
)

# Filter to only traces that have both inputs and outputs (skip incomplete ones)
valid_traces = [t for t in traces if t.inputs and t.outputs]
if valid_traces:
    ls_client.create_examples(
        dataset_id=trace_dataset.id,
        inputs=[t.inputs for t in valid_traces],    # trace inputs become example inputs
        outputs=[t.outputs for t in valid_traces],   # trace outputs become expected outputs
    )

print(f"Converted {len(valid_traces)} traces into '{trace_dataset.name}'")

/var/folders/xh/567fcvbs1gs5gqz64rfth0pm0000gn/T/ipykernel_55987/3058376758.py:1: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  traces = list(ls_client.list_runs(


Converted 20 traces into 'product-reviews-from-traces'


This is the trace-to-dataset pattern: query your traces, then map their inputs and outputs into examples. This is the most common way datasets are built in practice.

In [8]:
import pandas as pd
import ast

# Build a small DataFrame with input/output columns (simulates a real export)
csv_data = pd.DataFrame({
    "input": [
        {"review_text": "The 'Standing Desk' is sturdy. Five stars."},
        {"review_text": "My 'Air Purifier' rattles. Two stars."},
        {"review_text": "The 'Travel Mug' keeps coffee hot. Four stars."},
    ],
    "output": [
        {"product": "Standing Desk", "rating": 5, "sentiment": "positive"},
        {"product": "Air Purifier", "rating": 2, "sentiment": "negative"},
        {"product": "Travel Mug", "rating": 4, "sentiment": "positive"},
    ],
})

# Write to CSV, then read it back (mimics importing from an external source)
csv_data.to_csv("reviews.csv", index=False)
csv_df = pd.read_csv("reviews.csv")

# Create a new dataset for the CSV-imported examples
csv_dataset = ls_client.create_dataset(
    dataset_name="product-reviews-csv",
    description="Dataset imported from a CSV file"
)

# Pandas stores dicts as strings in CSV — convert them back to actual dicts
csv_inputs = [ast.literal_eval(row["input"]) for _, row in csv_df.iterrows()]
csv_outputs = [ast.literal_eval(row["output"]) for _, row in csv_df.iterrows()]

# Import all rows as examples in one batch
ls_client.create_examples(
    dataset_id=csv_dataset.id,
    inputs=csv_inputs,
    outputs=csv_outputs,
)

print(f"Imported {len(csv_df)} examples from CSV into '{csv_dataset.name}'")

Imported 3 examples from CSV into 'product-reviews-csv'


This writes a small CSV, reads it back, and imports each row as a dataset example. In practice, you'd export this from an existing system or spreadsheet.

In [9]:
# List all examples in the CSV dataset
all_examples = list(ls_client.list_examples(dataset_id=csv_dataset.id))

# Calculate 80/20 train/test split
split_idx = int(len(all_examples) * 0.8)

# Tag each example with its split using update_example()
for ex in all_examples[:split_idx]:
    ls_client.update_example(example_id=ex.id, split="train")
for ex in all_examples[split_idx:]:
    ls_client.update_example(example_id=ex.id, split="test")

print(f"Split {split_idx} into train, {len(all_examples) - split_idx} into test")

Split 2 into train, 1 into test


Splits let you organize examples into named subsets. When you run an evaluation, you can target a specific split (e.g., only run on test data).

In [10]:
# Re-read the dataset to verify everything looks correct
csv_dataset = ls_client.read_dataset(dataset_name="product-reviews-csv")
all_examples = list(ls_client.list_examples(dataset_id=csv_dataset.id))

# Filter examples by their split metadata
train = [e for e in all_examples if e.metadata.get("dataset_split", ["base"])[0] == "train"]
test = [e for e in all_examples if e.metadata.get("dataset_split", ["base"])[0] == "test"]

print(f"Dataset: {csv_dataset.name}")
print(f"  Total examples: {len(all_examples)}")
print(f"  Train: {len(train)} | Test: {len(test)}")

# Show a sample to confirm inputs/outputs look right
sample = all_examples[0]
print(f"\nSample (ID: {sample.id}):")
print(f"  Input: {sample.inputs}")
print(f"  Output: {sample.outputs}")
print(f"  Split: {sample.metadata.get('dataset_split', ['base'])[0]}")

Dataset: product-reviews-csv
  Total examples: 3
  Train: 2 | Test: 1

Sample (ID: c55806d5-51b7-43f8-b6ec-e2cef821098a):
  Input: {'review_text': "The 'Travel Mug' keeps coffee hot. Four stars."}
  Output: {'rating': 4, 'product': 'Travel Mug', 'sentiment': 'positive'}
  Split: test


This prints a summary of your dataset: total examples, count per split, and a sample.

## Verify in the LangSmith UI

Open [https://smith.langchain.com](https://smith.langchain.com) and check the following:

**Datasets tab** (left sidebar → Datasets):
- `product-reviews` — should show 13 examples (10 generated + 3 manual). Click into it to see each input/output pair.
- `product-reviews-from-traces` — examples converted from your LLM traces.
- `product-reviews-csv` — should show 3 examples imported from CSV, with train/test split labels visible.

**Traces tab** (left sidebar → your project `lab-3-datasets`):
- You should see LLM traces from Step 4 (the structured-output agent runs on 10 reviews). Each trace shows the input review text and the structured output.

**What to click:**
1. Go to **Datasets** → click `product-reviews` → you'll see a table of all 13 examples with Input and Output columns
2. Click any example row to expand it — verify the input is review text and the output is a `{product, rating, sentiment}` dict
3. Go back to Datasets → click `product-reviews-csv` → check that the Split column shows "train" or "test" for each example
4. Go to **Traces** → find your project → click any LLM trace → you should see the review text as input and the model's structured response as output

If all three datasets appear with correct inputs/outputs, Lab 3 is complete. These datasets are consumed by Lab 5 (Writing Custom Evaluators) and Lab 6 (Running Experiments).

## Optional Exercise

Create a third dataset by writing 10 movie descriptions (similar to the `Movie` schema from LangChain Lab 3) to a CSV file with columns `input` and `output`, then import it into LangSmith using the CSV import pattern from Step 8. Organize the examples into a `test` split and verify the dataset appears in your LangSmith workspace.